In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. CẤU HÌNH THÔNG SỐ
model_name = "wonrax/phobert-base-vietnamese-sentiment"
# Lưu ý: Model này dùng label: 0: NEG, 1: POS, 2: NEU
id2label = {0: "NEG", 1: "POS", 2: "NEU"}
label2id = {"NEG": 0, "POS": 1, "NEU": 2}

# 2. LOAD TOKENIZER VÀ MODEL GỐC
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

# 3. CẤU HÌNH LoRA (PEFT)
# Chúng ta chỉ train một nhóm nhỏ tham số "adapter" thay vì toàn bộ model
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, # Bài toán Phân loại văn bản
    r=8,                        # Rank (8 hoặc 16 là phổ biến)
    lora_alpha=32,
    target_modules=["query", "value"], # Áp dụng LoRA vào các lớp Attention
    lora_dropout=0.1,
    bias="none"
)

# Chuyển đổi model gốc thành model hỗ trợ LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters() 
# Bạn sẽ thấy chỉ có khoảng ~1% tham số được huấn luyện, cực kỳ nhẹ!

# 4. CHUẨN BỊ DATASET (PYTORCH)
class SentimentDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.comments = df['comment'].values
        self.labels = df['label_id'].values
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.comments)

    def __getitem__(self, index):
        encoding = self.tokenizer(
            str(self.comments[index]),
            truncation=True,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[index], dtype=torch.long)
        }

# Giả sử 'df' là DataFrame đã làm sạch của bạn
# Chia 80% train, 20% validation
df_train = pd.read_csv("/kaggle/input/datasets/ncmanh/data-for-phobert/data_preprocessed/Train_preprocessed.csv")
df_test = pd.read_csv("/kaggle/input/datasets/ncmanh/data-for-phobert/data_preprocessed/Test_preprocessed.csv")
df_val = pd.read_csv("/kaggle/input/datasets/ncmanh/data-for-phobert/data_preprocessed/Dev_preprocessed.csv")
train_dataset = SentimentDataset(df_train, tokenizer)
val_dataset = SentimentDataset(df_val, tokenizer)

# 5. THIẾT LẬP THAM SỐ HUẤN LUYỆN
training_args = TrainingArguments(
    output_dir="./phobert-lora-sentiment",
    learning_rate=2e-4,              # LoRA thường dùng LR cao hơn fine-tune truyền thống
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,              # Vì LoRA nhẹ nên có thể train nhiều epoch hơn
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
)

# 6. KHỞI TẠO TRAINER VÀ CHẠY
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("🚀 Bắt đầu huấn luyện với LoRA...")
trainer.train()

# 7. LƯU MODEL ADAPTER
model.save_pretrained("./final_shopee_model_lora")
tokenizer.save_pretrained("./final_shopee_model_lora")
print("✅ Đã lưu model thành công!")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

def evaluate_model(trainer, test_dataset, id2label):
    print("--- 📊 ĐANG TIẾN HÀNH ĐÁNH GIÁ TRÊN TẬP TEST ---")
    
    # 1. Dự đoán trên tập test
    # predictions sẽ chứa các giá trị logit (xác suất chưa chuẩn hóa)
    predictions, labels, _ = trainer.predict(test_dataset)
    
    # 2. Lấy chỉ số có xác suất cao nhất (Argmax)
    preds = np.argmax(predictions, axis=1)

    # 3. Tính Accuracy tổng quát
    acc = accuracy_score(labels, preds)
    print(f"✅ Accuracy Score: {acc*100:.2f}%")
    print("\n--- 📝 BÁO CÁO CHI TIẾT (CLASSIFICATION REPORT) ---")
    
    # Lấy danh sách tên nhãn theo đúng thứ tự 0, 1, 2
    target_names = [id2label[i] for i in range(len(id2label))]
    
    print(classification_report(labels, preds, target_names=target_names))

    # 4. Vẽ Ma trận nhầm lẫn (Confusion Matrix)
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=target_names, 
                yticklabels=target_names)
    plt.xlabel('Dự đoán (Predicted)')
    plt.ylabel('Thực tế (Actual)')
    plt.title('Confusion Matrix')
    plt.show()

# --- THỰC THI ĐÁNH GIÁ ---
# Giả sử bạn đã có df_test (phần dữ liệu tách riêng ra từ đầu chưa train)
test_dataset = SentimentDataset(df_test, tokenizer)

evaluate_model(trainer, test_dataset, id2label)